# 01 - Multimodal dual-stream attention

**학습 목표**: vision token은 clean stream에 한 번만 두고, noisy response는 같은 block과 이전 clean context만 보게 하는 Fast-dVLM식 toy mask를 만듭니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
vision = ['V0', 'V1']
clean = ['C0', 'C1', 'C2', 'C3']
noisy = ['N2', 'N3']  # response의 현재 block에 대응
labels = vision + clean + noisy

def clean_position(label):
    if label.startswith('V'):
        return int(label[1:])
    if label.startswith('C'):
        return len(vision) + int(label[1:])
    raise ValueError(label)

def allowed(query, key):
    if query.startswith(('V', 'C')):
        # clean stream은 token-level causal입니다.
        return key.startswith(('V', 'C')) and clean_position(key) <= clean_position(query)
    if query.startswith('N'):
        qpos = int(query[1:])
        if key.startswith('N'):
            return True  # 현재 noisy block 내부 bidirectional
        if key.startswith('V'):
            return True  # vision은 clean stream에서만 참조
        return key.startswith('C') and int(key[1:]) < qpos
    return False

matrix = [[allowed(q, k) for k in labels] for q in labels]
print('     ' + ' '.join(f'{x:>2}' for x in labels))
for q, row in zip(labels, matrix):
    print(f'{q:>3}: ' + '  '.join('1' if x else '.' for x in row))
assert allowed('N2', 'V1') and not allowed('N2', 'C2')

In [ ]:
def response_blocks(length, block_size):
    # 마지막 block을 response 경계에서 잘라 미래 turn 누수를 막습니다.
    return [list(range(start, min(start + block_size, length))) for start in range(0, length, block_size)]

print(response_blocks(length=7, block_size=4))
assert response_blocks(7, 4)[-1] == [4, 5, 6]

Vision token을 noisy stream에도 복제하면 같은 정보를 두 번 처리합니다. Fast-dVLM은 복제를 없애고 noisy token이 clean vision을 보게 해 lossless하게 memory/time을 줄였다고 보고합니다.